In [1]:
# Imports
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
import ipywidgets as widgets

In [2]:
# Shared paths & thresholds — from config.py
import sys
sys.path.insert(0, str(Path("..").resolve()))
from config import (
    ASSETS_PATH, ASSET_DAILY_PATH,
    MIN_VOLUME_24H_USD,
    DATE_COL as DATE_COLUMN,
    VOLUME_COL as VOLUME_COLUMN,
)

# Notebook-specific configuration
LOOKBACK_DAYS = 7

# Fixed economic basket thresholds — tuned per dataset, stay here
BASKET_4_UPPER = 100_000_000
BASKET_3_UPPER = 250_000_000
BASKET_2_UPPER = 1_000_000_000

In [3]:
# Core Helpers
def load_table(table_path: Path) -> pd.DataFrame:
    if not table_path.exists():
        raise FileNotFoundError(f"No data stored at {table_path}")

    return pd.read_parquet(table_path)


def get_asset_id_column(df: pd.DataFrame) -> str:
    possible_id_columns = ["asset_id", "coingecko_id", "id"]

    for column in possible_id_columns:
        if column in df.columns:
            return column

    raise KeyError(
        "No asset identifier column found. Expected one of: "
        f"{possible_id_columns}"
    )


def add_asset_metadata(
    df: pd.DataFrame,
    assets_df: pd.DataFrame,
    *,
    id_column: str,
) -> pd.DataFrame:
    metadata_columns = [id_column, "symbol", "name"]

    missing_columns = [
        column for column in metadata_columns
        if column not in assets_df.columns
    ]

    if missing_columns:
        raise KeyError(
            "Missing metadata columns in assets_df: "
            f"{missing_columns}"
        )

    asset_metadata_df = (
        assets_df[metadata_columns]
        .drop_duplicates(subset=[id_column])
        .copy()
    )

    return df.merge(
        asset_metadata_df,
        on=id_column,
        how="left",
    )


def compute_rolling_volume_table(
    asset_daily_df: pd.DataFrame,
    *,
    id_column: str,
    date_column: str,
    volume_column: str,
    lookback_days: int,
) -> pd.DataFrame:
    volume_df = asset_daily_df[[id_column, date_column, volume_column]].copy()
    volume_df[date_column] = pd.to_datetime(volume_df[date_column])
    volume_df = volume_df.sort_values([id_column, date_column])

    volume_df["rolling_median_volume_24h_usd"] = (
        volume_df
        .groupby(id_column)[volume_column]
        .transform(
            lambda series: series
            .rolling(lookback_days, min_periods=lookback_days)
            .median()
        )
    )

    volume_df["log_rolling_median_volume_24h_usd"] = np.log10(
        volume_df["rolling_median_volume_24h_usd"].where(
            volume_df["rolling_median_volume_24h_usd"] > 0
        )
    )

    return volume_df


def nice_volume_threshold(value: float) -> int | None:
    if pd.isna(value):
        return None

    magnitude = 10 ** np.floor(np.log10(value))
    normalized = value / magnitude

    if normalized <= 1:
        nice = 1
    elif normalized <= 2.5:
        nice = 2.5
    elif normalized <= 5:
        nice = 5
    else:
        nice = 10

    return int(nice * magnitude)


def suggest_fixed_thresholds(
    rolling_volume_df: pd.DataFrame,
    *,
    min_volume_24h_usd: float,
) -> dict:
    eligible_df = rolling_volume_df.loc[
        rolling_volume_df["rolling_median_volume_24h_usd"] >= min_volume_24h_usd
    ].copy()

    raw_thresholds = eligible_df["rolling_median_volume_24h_usd"].quantile(
        [0.25, 0.50, 0.75]
    )

    return {
        "basket_4_upper": nice_volume_threshold(raw_thresholds.loc[0.25]),
        "basket_3_upper": nice_volume_threshold(raw_thresholds.loc[0.50]),
        "basket_2_upper": nice_volume_threshold(raw_thresholds.loc[0.75]),
    }


def assign_fixed_volume_basket(
    volume: float,
    *,
    min_volume_24h_usd: float,
    basket_4_upper: float,
    basket_3_upper: float,
    basket_2_upper: float,
) -> str | None:
    if pd.isna(volume) or volume < min_volume_24h_usd:
        return None

    if volume < basket_4_upper:
        return "Basket 4"

    if volume < basket_3_upper:
        return "Basket 3"

    if volume < basket_2_upper:
        return "Basket 2"

    return "Basket 1"


def build_fixed_threshold_baskets(
    rolling_volume_df: pd.DataFrame,
    selected_date,
    *,
    id_column: str,
    date_column: str,
    min_volume_24h_usd: float,
    basket_4_upper: float,
    basket_3_upper: float,
    basket_2_upper: float,
) -> pd.DataFrame:
    selected_date = pd.Timestamp(selected_date)

    basket_df = rolling_volume_df.loc[
        rolling_volume_df[date_column] == selected_date
    ].copy()

    basket_df["basket"] = basket_df["rolling_median_volume_24h_usd"].apply(
        lambda volume: assign_fixed_volume_basket(
            volume,
            min_volume_24h_usd=min_volume_24h_usd,
            basket_4_upper=basket_4_upper,
            basket_3_upper=basket_3_upper,
            basket_2_upper=basket_2_upper,
        )
    )

    basket_df = (
        basket_df
        .loc[basket_df["basket"].notna()]
        .sort_values("rolling_median_volume_24h_usd", ascending=False)
        .reset_index(drop=True)
    )

    return basket_df


def compare_baskets(
    previous_basket_df: pd.DataFrame,
    current_basket_df: pd.DataFrame,
    *,
    id_column: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    previous_basket_map = previous_basket_df.set_index(id_column)["basket"]

    comparison_df = current_basket_df.copy()
    comparison_df["previous_basket"] = comparison_df[id_column].map(
        previous_basket_map
    )

    comparison_df["basket_change"] = np.select(
        [
            comparison_df["previous_basket"].isna(),
            comparison_df["previous_basket"].astype(str)
            != comparison_df["basket"].astype(str),
        ],
        [
            "entered universe",
            "switched basket",
        ],
        default="unchanged",
    )

    previous_assets = set(previous_basket_df[id_column])
    current_assets = set(current_basket_df[id_column])
    left_assets = previous_assets - current_assets

    left_df = (
        previous_basket_df
        .loc[
            previous_basket_df[id_column].isin(left_assets),
            [
                id_column,
                "basket",
                "rolling_median_volume_24h_usd",
            ],
        ]
        .rename(columns={"basket": "previous_basket"})
        .copy()
    )

    left_df["basket_change"] = "left universe"

    transition_df = (
        comparison_df
        .loc[comparison_df["previous_basket"].notna()]
        .groupby(["previous_basket", "basket"], observed=True)
        .size()
        .rename("asset_count")
        .reset_index()
        .pivot(index="previous_basket", columns="basket", values="asset_count")
        .fillna(0)
        .astype(int)
    )

    return comparison_df, left_df, transition_df


def build_basket_comparison(
    rolling_volume_df: pd.DataFrame,
    *,
    previous_date,
    selected_date,
    id_column: str,
    date_column: str,
    min_volume_24h_usd: float,
    basket_4_upper: float,
    basket_3_upper: float,
    basket_2_upper: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    previous_basket_df = build_fixed_threshold_baskets(
        rolling_volume_df,
        previous_date,
        id_column=id_column,
        date_column=date_column,
        min_volume_24h_usd=min_volume_24h_usd,
        basket_4_upper=basket_4_upper,
        basket_3_upper=basket_3_upper,
        basket_2_upper=basket_2_upper,
    )

    current_basket_df = build_fixed_threshold_baskets(
        rolling_volume_df,
        selected_date,
        id_column=id_column,
        date_column=date_column,
        min_volume_24h_usd=min_volume_24h_usd,
        basket_4_upper=basket_4_upper,
        basket_3_upper=basket_3_upper,
        basket_2_upper=basket_2_upper,
    )

    comparison_df, left_df, transition_df = compare_baskets(
        previous_basket_df,
        current_basket_df,
        id_column=id_column,
    )

    return (
        previous_basket_df,
        current_basket_df,
        comparison_df,
        left_df,
        transition_df,
    )

In [4]:
# Load Data
assets_df = load_table(ASSETS_PATH)
asset_daily_df = load_table(ASSET_DAILY_PATH)

asset_daily_df[DATE_COLUMN] = pd.to_datetime(asset_daily_df[DATE_COLUMN])

id_column = get_asset_id_column(asset_daily_df)

rolling_volume_df = compute_rolling_volume_table(
    asset_daily_df,
    id_column=id_column,
    date_column=DATE_COLUMN,
    volume_column=VOLUME_COLUMN,
    lookback_days=LOOKBACK_DAYS,
)

available_dates = sorted(rolling_volume_df[DATE_COLUMN].dropna().unique())
available_dates = [pd.Timestamp(date) for date in available_dates]

print(f"asset_daily_df: {asset_daily_df.shape}")
print(f"assets_df: {assets_df.shape}")
print(f"Asset ID column: {id_column}")
print(f"Dates: {available_dates[0].date()} to {available_dates[-1].date()}")

asset_daily_df: (29523, 5)
assets_df: (57, 9)
Asset ID column: asset_id
Dates: 2024-07-23 to 2026-07-23


In [ ]:
# Suggest Fixed Thresholds
threshold_suggestions = suggest_fixed_thresholds(
    rolling_volume_df,
    min_volume_24h_usd=MIN_VOLUME_24H_USD,
)

print("Suggested thresholds based on 7-day median volume quantiles:")
print(f"Basket 4 upper: ${threshold_suggestions['basket_4_upper']:,.0f}")
print(f"Basket 3 upper: ${threshold_suggestions['basket_3_upper']:,.0f}")
print(f"Basket 2 upper: ${threshold_suggestions['basket_2_upper']:,.0f}")

print("\nCurrent fixed basket definitions:")
print(f"Basket 1: volume >= ${BASKET_2_UPPER:,.0f}")
print(f"Basket 2: ${BASKET_3_UPPER:,.0f} <= volume < ${BASKET_2_UPPER:,.0f}")
print(f"Basket 3: ${BASKET_4_UPPER:,.0f} <= volume < ${BASKET_3_UPPER:,.0f}")
print(f"Basket 4: ${MIN_VOLUME_24H_USD:,.0f} <= volume < ${BASKET_4_UPPER:,.0f}")

Suggested thresholds based on 7-day median volume quantiles:
Basket 4 upper: $250,000,000
Basket 3 upper: $500,000,000
Basket 2 upper: $1,000,000,000

Current fixed basket definitions:
Basket 1: volume >= $1,000,000,000
Basket 2: $250,000,000 <= volume < $1,000,000,000
Basket 3: $100,000,000 <= volume < $250,000,000
Basket 4: $50,000,000 <= volume < $100,000,000


: 

In [ ]:
# Plot Volume Distribution With Fixed Thresholds
eligible_volume_df = rolling_volume_df.loc[
    rolling_volume_df["rolling_median_volume_24h_usd"] >= MIN_VOLUME_24H_USD
].copy()

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    eligible_volume_df["log_rolling_median_volume_24h_usd"].dropna(),
    bins=30,
    edgecolor="white",
    linewidth=0.8,
    alpha=0.85,
)

for threshold in [
    MIN_VOLUME_24H_USD,
    BASKET_4_UPPER,
    BASKET_3_UPPER,
    BASKET_2_UPPER,
]:
    ax.axvline(
        np.log10(threshold),
        linestyle="--",
        linewidth=1.5,
        label=f"${threshold:,.0f}",
    )

ax.set_title("Distribution of 7-Day Median USD Volume")
ax.set_xlabel("log10(7-day median 24h USD volume)")
ax.set_ylabel("Frequency")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)

fig.tight_layout()
plt.show()

In [ ]:
# Basket Analysis Helpers
def summarize_basket_counts(basket_df: pd.DataFrame) -> pd.DataFrame:
    return (
        basket_df
        .groupby("basket", observed=True)
        .size()
        .rename("asset_count")
        .reset_index()
        .sort_values("basket")
    )


def summarize_basket_volume(
    basket_df: pd.DataFrame,
    *,
    id_column: str,
) -> pd.DataFrame:
    return (
        basket_df
        .groupby("basket", observed=True)
        .agg(
            asset_count=(id_column, "count"),
            min_volume=("rolling_median_volume_24h_usd", "min"),
            median_volume=("rolling_median_volume_24h_usd", "median"),
            mean_volume=("rolling_median_volume_24h_usd", "mean"),
            max_volume=("rolling_median_volume_24h_usd", "max"),
        )
        .reset_index()
        .sort_values("basket")
    )


def summarize_changes(
    comparison_df: pd.DataFrame,
    left_df: pd.DataFrame,
) -> pd.DataFrame:
    current_changes = (
        comparison_df["basket_change"]
        .value_counts()
        .rename_axis("change_type")
        .reset_index(name="asset_count")
    )

    if left_df.empty:
        return current_changes

    left_count = pd.DataFrame(
        {
            "change_type": ["left universe"],
            "asset_count": [len(left_df)],
        }
    )

    return pd.concat(
        [current_changes, left_count],
        ignore_index=True,
    )


def get_changed_assets(
    comparison_df: pd.DataFrame,
    left_df: pd.DataFrame,
    *,
    assets_df: pd.DataFrame,
    id_column: str,
) -> pd.DataFrame:
    output_columns = [
        id_column,
        "symbol",
        "name",
        "previous_basket",
        "basket",
        "rolling_median_volume_24h_usd",
        "basket_change",
    ]

    entered_or_switched_df = (
        comparison_df
        .loc[
            comparison_df["basket_change"].isin(
                ["entered universe", "switched basket"]
            ),
            [
                id_column,
                "previous_basket",
                "basket",
                "rolling_median_volume_24h_usd",
                "basket_change",
            ],
        ]
        .copy()
    )

    if not left_df.empty:
        left_assets_df = left_df.copy()
        left_assets_df["basket"] = pd.NA

        entered_or_switched_df = pd.concat(
            [
                entered_or_switched_df,
                left_assets_df[
                    [
                        id_column,
                        "previous_basket",
                        "basket",
                        "rolling_median_volume_24h_usd",
                        "basket_change",
                    ]
                ],
            ],
            ignore_index=True,
        )

    entered_or_switched_df = add_asset_metadata(
        entered_or_switched_df,
        assets_df,
        id_column=id_column,
    )

    return entered_or_switched_df[output_columns]


def get_current_basket_members(
    comparison_df: pd.DataFrame,
    assets_df: pd.DataFrame,
    *,
    id_column: str,
) -> pd.DataFrame:
    basket_members_df = add_asset_metadata(
        comparison_df,
        assets_df,
        id_column=id_column,
    )

    basket_members_df = basket_members_df.sort_values(
        "rolling_median_volume_24h_usd",
        ascending=False,
    )

    return basket_members_df[
        [
            id_column,
            "symbol",
            "name",
            "basket",
            "previous_basket",
            "rolling_median_volume_24h_usd",
            "basket_change",
        ]
    ]


def plot_basket_counts(
    basket_counts_df: pd.DataFrame,
    selected_date: pd.Timestamp,
):
    fig, ax = plt.subplots(figsize=(7, 4))

    ax.bar(
        basket_counts_df["basket"].astype(str),
        basket_counts_df["asset_count"],
    )

    ax.set_title(f"Basket Sizes on {selected_date.date()}")
    ax.set_xlabel("Basket")
    ax.set_ylabel("Number of assets")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.25)

    fig.tight_layout()
    plt.show()


def plot_basket_volume_boxplot(
    basket_df: pd.DataFrame,
    selected_date: pd.Timestamp,
):
    basket_order = ["Basket 1", "Basket 2", "Basket 3", "Basket 4"]

    data = [
        basket_df.loc[
            basket_df["basket"] == basket,
            "rolling_median_volume_24h_usd",
        ].dropna()
        for basket in basket_order
    ]

    labels = [
        basket
        for basket, values in zip(basket_order, data)
        if len(values) > 0
    ]

    data = [values for values in data if len(values) > 0]

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.boxplot(data, labels=labels)

    ax.set_title(f"Volume Distribution by Basket on {selected_date.date()}")
    ax.set_xlabel("Basket")
    ax.set_ylabel("7-day median 24h USD volume")
    ax.set_yscale("log")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.25)

    fig.tight_layout()
    plt.show()

In [ ]:
# Basket Comparison
date_slider = widgets.IntSlider(
    value=len(available_dates) - 1,
    min=LOOKBACK_DAYS,
    max=len(available_dates) - 1,
    step=1,
    description="Day",
    continuous_update=False,
)

previous_button = widgets.Button(
    description="Previous day",
    button_style="",
)

next_button = widgets.Button(
    description="Next day",
    button_style="",
)

output = widgets.Output()


def render_fixed_threshold_baskets(date_index):
    with output:
        clear_output(wait=True)

        selected_date = available_dates[date_index]
        previous_date = available_dates[date_index - 1]

        (
            previous_basket_df,
            current_basket_df,
            comparison_df,
            left_df,
            transition_df,
        ) = build_basket_comparison(
            rolling_volume_df,
            previous_date=previous_date,
            selected_date=selected_date,
            id_column=id_column,
            date_column=DATE_COLUMN,
            min_volume_24h_usd=MIN_VOLUME_24H_USD,
            basket_4_upper=BASKET_4_UPPER,
            basket_3_upper=BASKET_3_UPPER,
            basket_2_upper=BASKET_2_UPPER,
        )

        basket_counts_df = summarize_basket_counts(current_basket_df)

        basket_volume_summary_df = summarize_basket_volume(
            current_basket_df,
            id_column=id_column,
        )

        change_summary_df = summarize_changes(comparison_df, left_df)

        changed_assets_df = get_changed_assets(
            comparison_df,
            left_df,
            assets_df=assets_df,
            id_column=id_column,
        )

        current_basket_members_df = get_current_basket_members(
            comparison_df,
            assets_df,
            id_column=id_column,
        )

        print(f"Selected date: {selected_date.date()}")
        print(f"Compared with: {previous_date.date()}")
        print(f"Lookback: {LOOKBACK_DAYS} days")
        print(f"Minimum volume: ${MIN_VOLUME_24H_USD:,.0f}")
        print(f"Eligible assets: {len(current_basket_df)}")

        print("\nFixed basket definitions:")
        print(f"Basket 1: volume >= ${BASKET_2_UPPER:,.0f}")
        print(f"Basket 2: ${BASKET_3_UPPER:,.0f} <= volume < ${BASKET_2_UPPER:,.0f}")
        print(f"Basket 3: ${BASKET_4_UPPER:,.0f} <= volume < ${BASKET_3_UPPER:,.0f}")
        print(f"Basket 4: ${MIN_VOLUME_24H_USD:,.0f} <= volume < ${BASKET_4_UPPER:,.0f}")

        print("\nBasket counts:")
        display(basket_counts_df)

        plot_basket_counts(basket_counts_df, selected_date)

        print("\nBasket volume summary:")
        display(basket_volume_summary_df)

        plot_basket_volume_boxplot(current_basket_df, selected_date)

        print("\nChange summary:")
        display(change_summary_df)

        print("\nTransition matrix:")
        display(transition_df)

        print("\nChanged assets:")
        display(changed_assets_df)

        print("\nCurrent basket members:")
        display(current_basket_members_df)


def on_slider_change(change):
    if change["name"] == "value":
        render_fixed_threshold_baskets(change["new"])


def on_previous_button_clicked(button):
    if date_slider.value > date_slider.min:
        date_slider.value -= 1


def on_next_button_clicked(button):
    if date_slider.value < date_slider.max:
        date_slider.value += 1


date_slider.observe(on_slider_change, names="value")
previous_button.on_click(on_previous_button_clicked)
next_button.on_click(on_next_button_clicked)

display(
    widgets.VBox(
        [
            date_slider,
            widgets.HBox([previous_button, next_button]),
            output,
        ]
    )
)

render_fixed_threshold_baskets(date_slider.value)

In [ ]:
# Full Change Log Across All Dates
change_log_rows = []

for date_index in range(LOOKBACK_DAYS, len(available_dates)):
    selected_date = available_dates[date_index]
    previous_date = available_dates[date_index - 1]

    (
        _previous_basket_df,
        _current_basket_df,
        comparison_df,
        left_df,
        _transition_df,
    ) = build_basket_comparison(
        rolling_volume_df,
        previous_date=previous_date,
        selected_date=selected_date,
        id_column=id_column,
        date_column=DATE_COLUMN,
        min_volume_24h_usd=MIN_VOLUME_24H_USD,
        basket_4_upper=BASKET_4_UPPER,
        basket_3_upper=BASKET_3_UPPER,
        basket_2_upper=BASKET_2_UPPER,
    )

    changed_current_df = comparison_df.loc[
        comparison_df["basket_change"].isin(
            ["entered universe", "switched basket"]
        )
    ].copy()

    for _, row in changed_current_df.iterrows():
        change_log_rows.append(
            {
                "date": selected_date,
                "previous_date": previous_date,
                id_column: row[id_column],
                "previous_basket": row["previous_basket"],
                "current_basket": row["basket"],
                "rolling_median_volume_24h_usd": row[
                    "rolling_median_volume_24h_usd"
                ],
                "change_type": row["basket_change"],
            }
        )

    for _, row in left_df.iterrows():
        change_log_rows.append(
            {
                "date": selected_date,
                "previous_date": previous_date,
                id_column: row[id_column],
                "previous_basket": row["previous_basket"],
                "current_basket": pd.NA,
                "rolling_median_volume_24h_usd": row[
                    "rolling_median_volume_24h_usd"
                ],
                "change_type": "left universe",
            }
        )

change_log_df = (
    pd.DataFrame(change_log_rows)
    .sort_values(["date", "change_type", id_column])
    .reset_index(drop=True)
)

change_log_df = add_asset_metadata(
    change_log_df,
    assets_df,
    id_column=id_column,
)

change_log_df = change_log_df[
    [
        "date",
        "previous_date",
        id_column,
        "symbol",
        "name",
        "previous_basket",
        "current_basket",
        "rolling_median_volume_24h_usd",
        "change_type",
    ]
]

print(f"Total changes: {len(change_log_df)}")

display(change_log_df)

Total changes: 5


,date,previous_date,asset_id,symbol,name,previous_basket,current_basket,rolling_median_volume_24h_usd,change_type
0,2026-01-15,2026-01-14,45,HBAR,Hedera,Basket 4,<NA>,50812250.0,left universe
1,2026-01-22,2026-01-21,45,HBAR,Hedera,NaN,Basket 4,50812250.0,entered universe
2,2026-02-04,2026-02-03,15,USDE,Ethena USDe,Basket 4,Basket 3,182822452.0,switched basket
3,2026-03-02,2026-03-01,45,HBAR,Hedera,Basket 4,<NA>,50946438.0,left universe
4,2026-03-05,2026-03-04,17,DAI,Dai,Basket 4,Basket 3,100161835.0,switched basket
